<a href="https://colab.research.google.com/github/ajinfajrian/DataScience_240401020100_Fajrian/blob/master/Pertemuan12_Fajrian_Ichlasul_240401020100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Nama:** Fajrian Ichlasul Maruto \
**NIM:** 240401020100 \
**Kelas:** IF401 \
**Mata Kuliah:** Data Science \
**Pertemuan 12:** Asosiasi Data & Sistem Rekomendasi Dasar

### Import Library & Generate Dataset Sintetis

In [7]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import random
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

# 1. Generate Katalog Produk
data_katalog = {
    'produk': ['Roti', 'Susu', 'Selai', 'Mentega', 'Kopi', 'Gula', 'Teh', 'Biskuit', 'Cokelat', 'Permen'],
    'kategori': ['Makanan Pokok', 'Minuman', 'Pelengkap', 'Pelengkap', 'Minuman', 'Bumbu', 'Minuman', 'Cemilan', 'Cemilan', 'Cemilan'],
    'harga': [15000, 20000, 25000, 18000, 30000, 12000, 15000, 10000, 22000, 5000]
}
katalog = pd.DataFrame(data_katalog)
print("--- Katalog Produk ---")
display(katalog.head())

# 2. Generate 50 Transaksi (Masing-masing 2-5 item)
np.random.seed(42)
random.seed(42)
transaksi = []
for i in range(50):
    jumlah_item = random.randint(2, 5)
    item_dibeli = random.sample(data_katalog['produk'], k=jumlah_item)
    # Membuat pola asosiasi buatan: Jika beli Roti, kemungkinan besar beli Selai/Susu
    if 'Roti' in item_dibeli and random.random() > 0.3:
        if 'Selai' not in item_dibeli: item_dibeli.append('Selai')
        if 'Susu' not in item_dibeli: item_dibeli.append('Susu')
    transaksi.append(list(set(item_dibeli))) # Hapus duplikat dalam 1 transaksi

print("\n--- Sampel 5 Transaksi Pertama ---")
for i, t in enumerate(transaksi[:5]):
    print(f"Transaksi {i+1}: {t}")

--- Katalog Produk ---


,produk,kategori,harga
0,Roti,Makanan Pokok,15000
1,Susu,Minuman,20000
2,Selai,Pelengkap,25000
3,Mentega,Pelengkap,18000
4,Kopi,Minuman,30000



--- Sampel 5 Transaksi Pertama ---
Transaksi 1: ['Roti', 'Kopi']
Transaksi 2: ['Permen', 'Cokelat', 'Susu']
Transaksi 3: ['Permen', 'Roti', 'Biskuit', 'Teh', 'Susu', 'Selai']
Transaksi 4: ['Cokelat', 'Mentega']
Transaksi 5: ['Roti', 'Biskuit', 'Kopi', 'Mentega', 'Teh', 'Susu', 'Selai']


### Market Basket Analysis (Algoritma Apriori)

In [8]:
# 1. Transformasi list transaksi menjadi format One-Hot Encoding (True/False)
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df_transaksi = pd.DataFrame(te_ary, columns=te.columns_)

# 2. Terapkan Algoritma Apriori (Mencari itemset dengan minimal support 20%)
frequent_itemsets = apriori(df_transaksi, min_support=0.2, use_colnames=True)

# 3. Ekstrak Association Rules (Aturan Asosiasi dengan confidence minimal 50%)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print("--- Hasil Association Rules (Market Basket Analysis) ---")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

--- Hasil Association Rules (Market Basket Analysis) ---


,antecedents,consequents,support,confidence,lift
0,(Roti),"(Selai, Susu)",0.28,0.736842,2.167183
1,"(Selai, Susu)",(Roti),0.28,0.823529,2.167183
2,"(Roti, Susu)",(Selai),0.28,0.875000,1.682692
3,(Selai),"(Roti, Susu)",0.28,0.538462,1.682692
4,"(Selai, Roti)",(Susu),0.28,1.000000,1.666667
5,(Selai),(Roti),0.28,0.538462,1.417004
6,(Roti),(Selai),0.28,0.736842,1.417004
7,(Susu),(Roti),0.32,0.533333,1.403509
8,(Roti),(Susu),0.32,0.842105,1.403509
9,(Biskuit),(Selai),0.20,0.714286,1.373626


### Sistem Rekomendasi (Content-Based Filtering)

In [9]:
# 1. Membuat representasi matriks dari kategori produk
cv = CountVectorizer()
kategori_matrix = cv.fit_transform(katalog['kategori'])

# 2. Menghitung Cosine Similarity antar produk
sim_matrix = cosine_similarity(kategori_matrix)

# 3. Fungsi Rekomendasi berdasarkan instruksi Modul
def rekomendasi_serupa(nama_produk, top_n=3):
    # Mencari index produk
    idx = katalog.index[katalog['produk'] == nama_produk][0]

    # Mengambil skor similarity dari produk tersebut terhadap semua produk lain
    skor = list(enumerate(sim_matrix[idx]))

    # Mengurutkan berdasarkan skor kemiripan tertinggi
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    # Mengambil top_n produk yang paling mirip (kecuali produk itu sendiri)
    skor = [s for s in skor if s[0] != idx][:top_n]

    # Mengembalikan daftar nama produk
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print("--- Hasil Sistem Rekomendasi ---")
produk_target = 'Roti'
print(f"Produk yang mirip dengan '{produk_target}':", rekomendasi_serupa(produk_target))

produk_target_2 = 'Teh'
print(f"Produk yang mirip dengan '{produk_target_2}':", rekomendasi_serupa(produk_target_2))

--- Hasil Sistem Rekomendasi ---
Produk yang mirip dengan 'Roti': ['Susu', 'Selai', 'Mentega']
Produk yang mirip dengan 'Teh': ['Susu', 'Kopi', 'Roti']


### Kesimpulan Pertemuan 12
* **Market Basket Analysis (Apriori):** Model berhasil menemukan pola keterkaitan antar produk dari data transaksi. Metrik *Lift* > 1 menunjukkan bahwa probabilitas pelanggan membeli produk konsekuen (misal: Selai) akan meningkat secara signifikan jika mereka sudah memasukkan produk anteseden (misal: Roti) ke dalam keranjang.
* **Sistem Rekomendasi (Content-Based):** Dengan memanfaatkan *Cosine Similarity* pada fitur kategori produk, sistem berhasil memberikan rekomendasi barang yang identik secara fungsi. Pendekatan ini sangat berguna untuk mengatasi masalah *cold-start* bagi pengguna baru yang belum memiliki riwayat (*user-item matrix*) yang cukup untuk *Collaborative Filtering*.